In [2]:
import pandas as pd
import os
import numpy as np
from datetime import datetime
from zoneinfo import ZoneInfo
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
pd.set_option('display.max_columns', None)

In [6]:
df_rep_errores= pd.read_excel('C:/DATOS/rep errores/PRODUCTO7753.xlsx', sheet_name=0,
                              dtype={'LOTES ANTERIORES': str, 'CODIGO PRODUCTO': str, 
                                     'CODIGO PLAN': str, 'COD DE CERTIFICADO': str, 
                                     'IDEDET': str, 'FECNACIMIENTO': str, 'TIPDOCUMENTO': str,
                                     'NUMDOCUMENTO':str, 'CODIGO ERROR':str})
df_rep_errores.columns = (df_rep_errores.columns.str.strip().str.upper()
                            .str.replace(r'[^A-Za-z0-9]', '_', regex=True))
df_rep_errores.drop([''], axis=1, inplace=True)

In [7]:
df_rep_errores= df_rep_errores.rename(columns={'COD_DE_CERTIFICADO':'CODIGO_CERTIFICADO',
                                               'FEC__INICIO':'FECHA_INICIO', 'FEC__FIN':'FECHA_FIN',
                                                'PRIMABRUTACAN':'PRIMA_BRUTA', 'PRIMANETACAN':'PRIMA_NETA',
                                                'NOMCOMPLETO': 'NOMBRE_COMPLETO','APEPATERNO': 'APELLIDO_PATERNO',
                                                'APEMATERNO': 'APELLIDO_MATERNO','FECNACIMIENTO':'FECHA_NACIMIENTO',
                                                'TIPDOCUMENTO':'TIPO_DOCUMENTO','NUMDOCUMENTO':'NUM_DOCUMENTO'})

In [9]:

df_rep_errores['TIPO_MOVIMIENTO'] = df_rep_errores["TIPO_MOVIMIENTO"].str.upper()
df_rep_errores['ORIGEN_ERROR'] = df_rep_errores["ORIGEN_ERROR"].str.upper()
df_rep_errores['FECHA_CARGA']= pd.to_datetime(df_rep_errores['FECHA_CARGA'], errors='coerce').dt.date
df_rep_errores['FECHA_NACIMIENTO']= pd.to_datetime(df_rep_errores['FECHA_NACIMIENTO'], format='%Y%m%d', errors='coerce').dt.date
df_rep_errores['FECHA_INICIO']= pd.to_datetime(df_rep_errores['FECHA_INICIO'], format='%d/%m/%Y', errors='coerce').dt.date
df_rep_errores['FECHA_FIN']= pd.to_datetime(df_rep_errores['FECHA_FIN'], format='%d/%m/%Y', errors='coerce').dt.date
df_rep_errores['SUMA_ASEGURADA'] = df_rep_errores['SUMA_ASEGURADA'].astype('Float64')
df_rep_errores['TASA'] = df_rep_errores['TASA'].astype('Float64')
df_rep_errores['TASA_RECARGO'] = df_rep_errores['TASA_RECARGO'].astype('Float64')
df_rep_errores= df_rep_errores.fillna({'LINEA_TRAMA': 'SIN DATO'})

In [10]:
df_rep_errores.loc[df_rep_errores['MONEDA'].str.lower().isin(['sol','pen']), 'MONEDA'] = 'SOL'

In [11]:
df_rep_errores["CODIGO_SEGURO"] = np.where(df_rep_errores["LINEA_TRAMA"] != "SIN DATO",
                                            df_rep_errores["LINEA_TRAMA"].str[:3],'SIN DATO')
df_rep_errores["CODIGO_MOVIMIENTO"] = np.where(df_rep_errores["LINEA_TRAMA"] != "SIN DATO",
                                            df_rep_errores["LINEA_TRAMA"].str[48],'SIN DATO')
   

In [12]:
df_rep_errores['CODIGO_ERROR'] = df_rep_errores['CODIGO_ERROR'].apply(lambda x: str(int(x)).zfill(4) if pd.notnull(x) else np.nan)

In [13]:
df_rep_errores['MONEDA'].value_counts()

MONEDA
SOL    1260
Name: count, dtype: int64

In [14]:
df_rep_errores.head(3)

,NRO_LOTE,LOTES_ANTERIORES,FECHA_CARGA,CODIGO_PRODUCTO,PRODUCTO,CODIGO_PLAN,NOMBRE_DE_PLAN,CODIGO_CERTIFICADO,IDEDET,TIPO_MOVIMIENTO,FECHA_INICIO,FECHA_FIN,MONEDA,SUMA_ASEGURADA,TASA,TASA_RECARGO,PRIMA_BRUTA,PRIMA_NETA,NOMBRE_COMPLETO,APELLIDO_PATERNO,APELLIDO_MATERNO,FECHA_NACIMIENTO,TIPO_DOCUMENTO,NUM_DOCUMENTO,NOMBRE_DE_ARCHIVO,LINEA_TRAMA,ORIGEN_ERROR,CODIGO_ERROR,DESCRIPCION_ERROR,CODIGO_SEGURO,CODIGO_MOVIMIENTO
0,1409449,NaN,2026-06-27,7753,SALUD FLEXIBLE FALABELLA,391225,SALUD FLEXIBLE FALABELLA JULIO 2022,00000000000009425238,905897949,ABONO,2026-06-22,2026-07-22,SOL,0.0,0.0,0.0,246,202.40,MELIZA ELIZABET,ROSALES,SALDIVAR,1980-01-22,2,40666393,20509608467_4137001_20260622_004.TXT,30200000000000009425238 01P...,ERROR CANAL,0199,No se puede ejecutar la validación VAL - Moned...,302,4
1,1409449,NaN,2026-06-27,7753,SALUD FLEXIBLE FALABELLA,391225,SALUD FLEXIBLE FALABELLA JULIO 2022,00000000000009425238,905897949,ABONO,2026-06-22,2026-07-22,SOL,0.0,0.0,0.0,246,202.40,MELIZA ELIZABET,ROSALES,SALDIVAR,1980-01-22,2,40666393,20509608467_4137001_20260622_004.TXT,30200000000000009425238 01P...,ERROR CANAL,1074,EL CERTIFICADO TIENE UN PAGO PENDIENTE DE CORR...,302,4
2,1409449,NaN,2026-06-27,7753,SALUD FLEXIBLE FALABELLA,391235,SALUD FLEXIBLE FALABELLA JULIO 2022,00000000000009790787,905897965,ABONO,2026-06-22,2026-07-22,SOL,0.0,0.0,0.0,248,204.05,CARMEN VANESSA,MAYHUASCA,CLEMENTE,1988-06-01,2,45307054,20509608467_4137001_20260622_004.TXT,30200000000000009790787 01P...,ERROR CANAL,0199,No se puede ejecutar la validación VAL - Moned...,302,4


In [15]:
df_rep_errores.info()

<class 'pandas.DataFrame'>
RangeIndex: 1260 entries, 0 to 1259
Data columns (total 31 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   NRO_LOTE            1260 non-null   int64  
 1   LOTES_ANTERIORES    324 non-null    str    
 2   FECHA_CARGA         1260 non-null   object 
 3   CODIGO_PRODUCTO     1260 non-null   str    
 4   PRODUCTO            1260 non-null   str    
 5   CODIGO_PLAN         1260 non-null   str    
 6   NOMBRE_DE_PLAN      1168 non-null   str    
 7   CODIGO_CERTIFICADO  1260 non-null   str    
 8   IDEDET              1260 non-null   str    
 9   TIPO_MOVIMIENTO     1260 non-null   str    
 10  FECHA_INICIO        1260 non-null   object 
 11  FECHA_FIN           1260 non-null   object 
 12  MONEDA              1260 non-null   str    
 13  SUMA_ASEGURADA      1260 non-null   Float64
 14  TASA                1260 non-null   Float64
 15  TASA_RECARGO        1260 non-null   Float64
 16  PRIMA_BRUTA      